<a href="https://colab.research.google.com/github/mallikaalvala/streamlit-my-app.py/blob/main/sdrg_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 SDRG & Define.xml QC Checker
---
This notebook performs **10 automated quality checks** on:
- 📄 **PDF** – Study Data Reviewer's Guide (SDRG)
- 🌐 **HTML** – define.xml rendered as HTML (Define 2.0)

| Check | Description |
|---|---|
| 1 | Date consistency between PDF and HTML |
| 2 | Variables with missing codelist name (HTML) |
| 3 | Section 4.2 Issues Summary (PDF) |
| 4 | Same variable name with different labels (HTML) |
| 5 | Keyword detection (PDF & HTML) |
| 6 | Unbalanced quotations (PDF & HTML) |
| 7 | Typo / spelling issues (PDF & HTML) |
| 8 | Page number integrity (PDF) |
| 9 | Title consistency (PDF ↔ HTML) |
| 10 | Appendix presence (PDF) |

## ⚙️ Step 0 – Install Dependencies

In [1]:
# Run this cell once to install required libraries
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pdfplumber', 'beautifulsoup4', 'pyspellchecker', 'lxml', 'pandas'])
print('✅ All dependencies installed.')

✅ All dependencies installed.


## 📦 Step 1 – Imports & Utility Functions

In [3]:
import re
import io
import difflib
import collections
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Markdown

import pdfplumber
from bs4 import BeautifulSoup

# ── Display helpers ──────────────────────────────────────────────────────────
def section(title, emoji='📌'):
    display(HTML(f'''
        <div style="background:#1a3c6e;color:white;padding:10px 16px;
                    border-radius:8px;font-size:1.1rem;font-weight:700;
                    margin-top:28px;margin-bottom:8px;">
            {emoji} {title}
        </div>
    '''))

def badge(status):
    colors = {
        'PASS': ('#c6f6d5','#22543d'),
        'WARN': ('#fefcbf','#744210'),
        'FAIL': ('#fed7d7','#742a2a'),
    }
    bg, fg = colors.get(status, ('#e2e8f0','#2d3748'))
    display(HTML(f'<span style="background:{bg};color:{fg};padding:4px 14px;border-radius:6px;font-weight:700;font-size:1rem;">Status: {status}</span><br><br>'))

def info(msg):
    display(HTML(f'<div style="background:#ebf8ff;border-left:4px solid #3182ce;"'
        f'padding:8px 14px;border-radius:4px;margin:6px 0;font-size:0.95rem;">{msg}</div>'))

def show_df(data, title=None):
    if title:
        display(Markdown(f'**{title}**'))
    if data:
        display(pd.DataFrame(data))
    else:
        display(HTML('<i style="color:green;">✅ No issues found.</i>'))

print('✅ Helpers loaded.')

✅ Helpers loaded.


## 📂 Step 2 – Configure File Paths
> **Edit the two paths below** to point to your PDF and HTML files.

In [4]:
# ── 🔧 SET YOUR FILE PATHS HERE ─────────────────────────────────────────────
PDF_PATH  = '/content/reviewersguide.pdf'               # Path to SDRG PDF
HTML_PATH = '/content/define2-0-0-example-sdtm.html'   # Path to define.xml HTML
# ─────────────────────────────────────────────────────────────────────────────

assert Path(PDF_PATH).exists(),  f'PDF not found:  {PDF_PATH}'
assert Path(HTML_PATH).exists(), f'HTML not found: {HTML_PATH}'
print(f'✅ PDF  : {PDF_PATH}')
print(f'✅ HTML : {HTML_PATH}')

✅ PDF  : /content/reviewersguide.pdf
✅ HTML : /content/define2-0-0-example-sdtm.html


## 🔄 Step 3 – Parse Files

In [5]:
# ── Parse PDF ────────────────────────────────────────────────────────────────
pdf_pages = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page in pdf.pages:
        pdf_pages.append(page.extract_text() or '')
full_pdf_text = '\n'.join(pdf_pages)

# ── Parse HTML ───────────────────────────────────────────────────────────────
with open(HTML_PATH, 'r', encoding='utf-8', errors='replace') as f:
    raw_html = f.read()
soup = BeautifulSoup(raw_html, 'html.parser')
html_text = soup.get_text(separator='\n')

# ── Extract variable rows from HTML tables ───────────────────────────────────
def get_variable_rows(soup):
    rows = []
    for table in soup.find_all('table'):
        headers = [th.get_text(strip=True) for th in table.find_all('th')]
        if 'Variable' not in headers:
            continue
        col_idx = {h: i for i, h in enumerate(headers)}
        for tr in table.find_all('tr')[1:]:
            cells = [td.get_text(strip=True) for td in tr.find_all('td')]
            if not cells:
                continue
            row = {col: (cells[idx] if idx < len(cells) else '') for col, idx in col_idx.items()}
            rows.append(row)
    return rows

var_rows = get_variable_rows(soup)

print(f'✅ PDF parsed   → {len(pdf_pages)} pages')
print(f'✅ HTML parsed  → {len(var_rows)} variable rows')

✅ PDF parsed   → 9 pages
✅ HTML parsed  → 555 variable rows


---
## ✅ Check 1 – Date Consistency (PDF ↔ HTML)

In [6]:
section('Check 1 – Date Consistency (PDF ↔ HTML)', '📅')

def extract_dates(text):
    patterns = [
        r'\b\d{4}-\d{2}-\d{2}\b',
        r'\b\d{2}/\d{2}/\d{4}\b',
        r'\b\d{4}-\d{2}-\d{2}T\d{2}:\d{2}',
    ]
    dates = []
    for p in patterns:
        dates.extend(re.findall(p, text))
    return sorted(set(dates))

pdf_dates  = extract_dates(full_pdf_text)
html_dates = extract_dates(html_text)
only_pdf   = sorted(set(pdf_dates) - set(html_dates))
only_html  = sorted(set(html_dates) - set(pdf_dates))
common     = sorted(set(pdf_dates) & set(html_dates))

print(f'  Dates in PDF  : {len(pdf_dates)}')
print(f'  Dates in HTML : {len(html_dates)}')
print(f'  Common dates  : {len(common)}')

show_df([{'Date': d} for d in only_pdf],  '🔶 Dates only in PDF')
show_df([{'Date': d} for d in only_html], '🔷 Dates only in HTML')
show_df([{'Date': d} for d in common],    '✅ Dates in both')
badge('WARN' if only_pdf or only_html else 'PASS')

  Dates in PDF  : 4
  Dates in HTML : 2
  Common dates  : 0


**🔶 Dates only in PDF**

,Date
0,2011-07-22
1,2012-07-01
2,2012-09-01
3,2014-08-06


**🔷 Dates only in HTML**

,Date
0,2013-03-03T17:04
1,2013-03-04


**✅ Dates in both**

---
## ✅ Check 2 – Variables Missing Codelist Name (HTML)

In [7]:
section('Check 2 – Variables Missing Codelist Name (HTML)', '📋')

missing_cl = []
for row in var_rows:
    ct_col = row.get('Controlled Terms or Format', '')
    var    = row.get('Variable', '')
    label  = row.get('Label', '')
    if var and ct_col.strip() == '':
        missing_cl.append({'Variable': var, 'Label': label, 'CT / Codelist': '(blank)'})

if missing_cl:
    print(f'⚠️  {len(missing_cl)} variable(s) have no controlled-term or codelist entry.')
    show_df(missing_cl, 'Variables Missing Codelist')
else:
    info('✅ All variables have a controlled-term / codelist entry.')
badge('WARN' if missing_cl else 'PASS')

⚠️  383 variable(s) have no controlled-term or codelist entry.


**Variables Missing Codelist**

,Variable,Label,CT / Codelist
0,STUDYID,Study Identifier,(blank)
1,TAETORD,Order of Element within Arm,(blank)
2,ETCD,Element Code,(blank)
3,ELEMENT,Description of Element,(blank)
4,TABRANCH,Branch,(blank)
...,...,...,...
378,VSORRES,,(blank)
379,VSORRES,,(blank)
380,VSORRES,,(blank)
381,VSORRES,,(blank)


---
## ✅ Check 3 – Section 4.2 Issues Summary (PDF)

In [8]:
section('Check 3 – Section 4.2 Issues Summary (PDF)', '📊')

pattern = r'4\.2\s+Issues?\s+Summary(.*?)(?=\n\s*(?:4\.\d|5\.|Appendix|$))'
match = re.search(pattern, full_pdf_text, re.IGNORECASE | re.DOTALL)

if match:
    section_text = match.group(1).strip()
    print('── Extracted Section 4.2 ──\n')
    print(section_text)
    print()
    issue_rows = []
    for line in section_text.splitlines():
        if re.search(r'\b(Error|Warning|Note|Info)\b', line, re.IGNORECASE):
            issue_rows.append({'Issue Line': line.strip()})
    show_df(issue_rows, 'Identified Issue Rows')
else:
    idx = full_pdf_text.lower().find('4.2')
    if idx != -1:
        print('Section 4.2 vicinity (heuristic):\n')
        print(full_pdf_text[idx: idx+800])
    else:
        print('❌ Section 4.2 not found in PDF.')
badge('PASS')

── Extracted Section 4.2 ──

............................................................................................................................ 8



**Identified Issue Rows**

---
## ✅ Check 4 – Same Variable Name, Different Labels (HTML)

In [9]:
section('Check 4 – Same Variable Name, Different Labels (HTML)', '🔀')

var_label_map = collections.defaultdict(set)
for row in var_rows:
    v = row.get('Variable', '').strip().upper()
    l = row.get('Label', '').strip()
    if v and l:
        var_label_map[v].add(l)

conflicts = {v: labels for v, labels in var_label_map.items() if len(labels) > 1}

if conflicts:
    print(f'⚠️  {len(conflicts)} variable(s) have inconsistent labels.')
    rows_out = [{'Variable': v, 'Labels Found': ' | '.join(sorted(l))}
                for v, l in sorted(conflicts.items())]
    show_df(rows_out, 'Label Conflicts')
else:
    info('✅ All variable names have consistent labels across all datasets.')
badge('FAIL' if conflicts else 'PASS')

⚠️  2 variable(s) have inconsistent labels.


**Label Conflicts**

,Variable,Labels Found
0,IETESTCD,Incl/Excl Criterion Short Name | Inclusion/Exc...
1,TAETORD,Order of Element within Arm | Planned Order of...


---
## ✅ Check 5 – Keyword Detection (PDF & HTML)

In [10]:
section('Check 5 – Keyword Detection (PDF & HTML)', '🔎')

KEYWORDS = ['please', 'path', 'update', 'required', 'cutoff date',
            'attachment', 'add', 'client', 'vendor', 'will']

def find_keyword_hits(text, source):
    hits = []
    for kw in KEYWORDS:
        for i, line in enumerate(text.splitlines(), 1):
            if re.search(r'\b' + re.escape(kw) + r'\b', line, re.IGNORECASE):
                hits.append({
                    'Source' : source,
                    'Keyword': kw,
                    'Line #' : i,
                    'Context': line.strip()[:150],
                })
    return hits

all_hits = find_keyword_hits(full_pdf_text, 'PDF') + find_keyword_hits(html_text, 'HTML')

if all_hits:
    print(f'⚠️  {len(all_hits)} keyword occurrence(s) detected.')
    show_df(all_hits, 'Keyword Hits')
else:
    info('✅ No restricted keywords found.')
badge('WARN' if all_hits else 'PASS')

⚠️  5 keyword occurrence(s) detected.


**Keyword Hits**

,Source,Keyword,Line #,Context
0,PDF,please,57,The trial inclusion/exclusion criteria are not...
1,PDF,will,91,Reference start date was assigned as the date ...
2,PDF,will,129,"subject completed Day 85, not all subjects wil..."
3,PDF,will,143,Temperature was not collected prior to randomi...
4,PDF,will,149,the investigator will record this as an advers...


---
## ✅ Check 6 – Unbalanced Quotations (PDF & HTML)

In [11]:
section('Check 6 – Unbalanced Quotations (PDF & HTML)', '❝')

def find_unbalanced_quotes(text, source):
    issues = []
    for i, line in enumerate(text.splitlines(), 1):
        dq      = line.count('"')
        sq_open  = line.count('\u2018')   # '
        sq_close = line.count('\u2019')   # '
        dq_open  = line.count('\u201c')   # "
        dq_close = line.count('\u201d')   # "
        if dq % 2 != 0:
            issues.append({'Source': source, 'Line': i,
                           'Issue': 'Odd straight double-quotes',
                           'Context': line.strip()[:120]})
        if sq_open != sq_close:
            issues.append({'Source': source, 'Line': i,
                           'Issue': 'Mismatched curly single-quotes',
                           'Context': line.strip()[:120]})
        if dq_open != dq_close:
            issues.append({'Source': source, 'Line': i,
                           'Issue': 'Mismatched curly double-quotes',
                           'Context': line.strip()[:120]})
    return issues

pdf_q  = find_unbalanced_quotes(full_pdf_text, 'PDF')
html_q = find_unbalanced_quotes(html_text,     'HTML')
all_q  = pdf_q + html_q

if all_q:
    print(f'⚠️  {len(all_q)} unbalanced quote issue(s) found.')
    show_df(all_q[:80], 'Quote Issues (first 80)')
else:
    info('✅ No unbalanced quotations found.')
badge('WARN' if all_q else 'PASS')

⚠️  13 unbalanced quote issue(s) found.


**Quote Issues (first 80)**

,Source,Line,Issue,Context
0,PDF,1,Mismatched curly single-quotes,Study Data Reviewer’s Guide
1,PDF,5,Mismatched curly single-quotes,Study SDRG-001A Study Data Reviewer’s Guide
2,PDF,6,Mismatched curly single-quotes,Study Data Reviewer’s Guide
3,PDF,32,Mismatched curly single-quotes,Study SDRG-001A Study Data Reviewer’s Guide
4,PDF,52,Mismatched curly single-quotes,Study SDRG-001A Study Data Reviewer’s Guide
5,PDF,63,Mismatched curly single-quotes,Study SDRG-001A Study Data Reviewer’s Guide
6,PDF,98,Mismatched curly single-quotes,Study SDRG-001A Study Data Reviewer’s Guide
7,PDF,126,Mismatched curly single-quotes,are expected. DSSCAT equal to TREATMENT DISCON...
8,PDF,128,Mismatched curly single-quotes,subject’s completion status at study exit. As ...
9,PDF,130,Mismatched curly single-quotes,Study SDRG-001A Study Data Reviewer’s Guide


---
## ✅ Check 7 – Typo / Spelling Issues (PDF & HTML)

In [12]:
section('Check 7 – Typo / Spelling Issues (PDF & HTML)', '✏️')

COMMON_WORDS = {
    'the','be','to','of','and','a','in','that','have','it','for','not','on','with',
    'as','you','do','at','this','but','by','from','they','we','say','she','or','an',
    'will','my','one','all','would','there','their','what','so','up','out','if',
    'about','who','get','which','go','me','when','make','can','like','time','no',
    'just','him','know','take','people','into','year','your','good','some','could',
    'them','see','other','than','then','now','look','only','come','its','over',
    'think','also','back','after','use','two','how','our','work','first','well',
    'even','new','want','because','any','these','give','day','most','us','study',
    'data','subject','domain','dataset','variable','label','type','value','origin',
    'sdtm','crf','visit','date','time','test','result','status','code','number',
    'name','description','class','key','length','format','controlled','terms',
    'derivation','version','protocol','period','week','dose','adverse','event',
    'submission','analysis','include','record','observation','missing','null',
    'assigned','collected','reported','defined','specified','based','related',
    'noted','expected','found','used','was','were','are','has','had','been','is',
    'does','did','done','may','might','must','shall','should','need','per','via',
    'both','between','within','during','prior','after','before','following','total',
    'additional','information','please','section','table','figure','appendix',
    'clinical','medical','subject','patient','investigator','sponsor','criteria'
}
MEDICAL_TERMS = {
    'sdtm','cdisc','sdrg','spirometry','bronchodilator','fev1','asthma',
    'exacerbation','subcutaneous','randomization','immunodeficiency','carcinoma',
    'placebo','corticosteroid','meddra','opencdisc','lbcvresc','lbcvresu',
    'lbcvnrlo','lbcvnrhi','dscat','dsscat','dsdecod','exoccur','vstestcd','blfl',
    'aeout','aedict','ietestcd','relrec','qnam','armcd','studyid','usubjid',
    'testcd','orres','orresu','nrlo','nrhi','stresc','stresu','stresn','visitnum',
    'visitdy','epoch','taetord','etcd','tabulation','tabulated'
}

try:
    from spellchecker import SpellChecker
    spell  = SpellChecker()
    combined = full_pdf_text + '\n' + html_text
    words  = re.findall(r'\b[a-zA-Z]{4,}\b', combined)
    unique = set(w.lower() for w in words) - COMMON_WORDS - MEDICAL_TERMS
    misspelled = spell.unknown(unique)
    typos = []
    for word in sorted(misspelled):
        cands = spell.candidates(word) or set()
        cands -= {word}
        typos.append({'Word': word, 'Suggestions': ', '.join(list(cands)[:4])})
    if typos:
        print(f'⚠️  {len(typos)} potential typo(s) detected.')
        show_df(typos, 'Potential Typos')
    else:
        info('✅ No spelling issues detected.')
    badge('WARN' if typos else 'PASS')
except ImportError:
    print('⚠️  pyspellchecker not installed. Run: pip install pyspellchecker')

⚠️  302 potential typo(s) detected.


**Potential Typos**

,Word,Suggestions
0,addon,"sadden, amon, ardor, deon"
1,aeacn,"aback, lean, react, alack"
2,aebodsys,
3,aedecod,
4,aeendtc,
...,...,...
297,whodrug,
298,withdrawel,withdrawal
299,ycaciffe,
300,ytefas,"stefan, yeas, teas, stelas"


---
## ✅ Check 8 – Page Number Integrity (PDF)

In [13]:
section('Check 8 – Page Number Integrity (PDF)', '📄')

page_pattern = re.compile(r'[Pp]age\s+(\d+)\s+of\s+(\d+)')
issues = []

for i, page_text in enumerate(pdf_pages, 1):
    for m in page_pattern.findall(page_text):
        current, total = int(m[0]), int(m[1])
        if current != i:
            issues.append({
                'Physical Page': i, 'Stated Page': current,
                'Stated Total': total,
                'Issue': f'Physical {i} ≠ stated {current}'
            })
        if total != len(pdf_pages):
            issues.append({
                'Physical Page': i, 'Stated Page': current,
                'Stated Total': total,
                'Issue': f'Total stated {total} but PDF has {len(pdf_pages)} pages'
            })

print(f'Total PDF pages: {len(pdf_pages)}')
# Deduplicate
seen, unique_issues = set(), []
for iss in issues:
    key = (iss['Physical Page'], iss['Stated Page'])
    if key not in seen:
        seen.add(key)
        unique_issues.append(iss)

if unique_issues:
    print(f'⚠️  {len(unique_issues)} page-number inconsistency/ies found.')
    show_df(unique_issues, 'Page Number Issues')
else:
    info('✅ Page numbers appear consistent throughout the PDF.')
badge('FAIL' if unique_issues else 'PASS')

Total PDF pages: 9


---
## ✅ Check 9 – Title Consistency (PDF ↔ HTML)

In [14]:
section('Check 9 – Title Consistency (PDF ↔ HTML)', '📝')

pdf_title_lines = [l.strip() for l in pdf_pages[0].splitlines() if l.strip()][:5]
pdf_title  = ' | '.join(pdf_title_lines)
html_title = soup.title.get_text(strip=True) if soup.title else 'Not found'
h1_tags    = [h.get_text(strip=True) for h in soup.find_all('h1')]

print(f'PDF first-page text : {pdf_title}')
print(f'HTML <title>        : {html_title}')
if h1_tags:
    print(f'HTML <h1> tag(s)    : {" | ".join(h1_tags[:3])}')

ratio = difflib.SequenceMatcher(None, pdf_title.lower(), html_title.lower()).ratio()
print(f'\nTitle similarity score: {ratio:.0%}')

if ratio > 0.5:
    info('✅ Titles appear broadly consistent (same study context).')
    badge('PASS')
else:
    info('⚠️ Titles differ significantly – verify study name alignment.')
    badge('WARN')

PDF first-page text : Study Data Reviewer’s Guide | SDRG, Inc. | Study SDRG-001A | Version 2014-08-06
HTML <title>        : Study CDISC01, Data Definitions
HTML <h1> tag(s)    : Tabulation Data Definition Tables | Value Level Metadata | Controlled Terms

Title similarity score: 35%


---
## ✅ Check 10 – Appendix Presence (PDF)

In [15]:
section('Check 10 – Appendix Presence (PDF)', '📎')

appendix_pattern = re.compile(r'\bAppendix\b', re.IGNORECASE)
pages_with_appendix = []

for i, page_text in enumerate(pdf_pages, 1):
    if appendix_pattern.search(page_text):
        for line in page_text.splitlines():
            if appendix_pattern.search(line):
                pages_with_appendix.append({'Page': i, 'Line': line.strip()[:150]})
                break

if pages_with_appendix:
    print(f'✅ Appendix found on {len(pages_with_appendix)} page(s).')
    show_df(pages_with_appendix, 'Appendix Locations')
    last_page = pages_with_appendix[-1]['Page']
    if last_page >= len(pdf_pages) - 2:
        info('📌 Appendix appears near the end of the document — as expected.')
    else:
        info(f'⚠️ Last appendix reference is on page {last_page} of {len(pdf_pages)}.')
else:
    print('❌ No Appendix section detected in the PDF.')
badge('PASS' if pages_with_appendix else 'FAIL')

✅ Appendix found on 3 page(s).


**Appendix Locations**

,Page,Line
0,2,Appendix I: Inclusion/Exclusion Criteria ........
1,4,Appendix I for the full text of the criteria.
2,9,Appendix I: Inclusion/Exclusion Criteria


---
## 📊 Final Summary

In [16]:
section('QC Summary Dashboard', '📊')

summary = [
    {'#': 1,  'Check': 'Date Consistency',              'Scope': 'PDF ↔ HTML'},
    {'#': 2,  'Check': 'Missing Codelist Name',         'Scope': 'HTML'},
    {'#': 3,  'Check': 'Section 4.2 Summary',           'Scope': 'PDF'},
    {'#': 4,  'Check': 'Same Variable Different Labels','Scope': 'HTML'},
    {'#': 5,  'Check': 'Keyword Detection',             'Scope': 'PDF & HTML'},
    {'#': 6,  'Check': 'Unbalanced Quotations',         'Scope': 'PDF & HTML'},
    {'#': 7,  'Check': 'Typo Detection',                'Scope': 'PDF & HTML'},
    {'#': 8,  'Check': 'Page Number Integrity',         'Scope': 'PDF'},
    {'#': 9,  'Check': 'Title Consistency',             'Scope': 'PDF ↔ HTML'},
    {'#': 10, 'Check': 'Appendix Presence',             'Scope': 'PDF'},
]
df_summary = pd.DataFrame(summary)
display(df_summary.style
    .set_properties(**{'text-align': 'left', 'font-size': '13px'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#1a3c6e'), ('color', 'white'),
                  ('font-weight', 'bold'), ('text-align', 'left')]
    }])
)
print('\n✅ All checks complete. Review individual cells above for detailed results.')

,#,Check,Scope
0,1,Date Consistency,PDF ↔ HTML
1,2,Missing Codelist Name,HTML
2,3,Section 4.2 Summary,PDF
3,4,Same Variable Different Labels,HTML
4,5,Keyword Detection,PDF & HTML
5,6,Unbalanced Quotations,PDF & HTML
6,7,Typo Detection,PDF & HTML
7,8,Page Number Integrity,PDF
8,9,Title Consistency,PDF ↔ HTML
9,10,Appendix Presence,PDF



✅ All checks complete. Review individual cells above for detailed results.
